In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import requests
import pandas as pd
import time
from dotenv import load_dotenv

# --- 1. Environment Setup ---
load_dotenv("../../config/.env") 
contact_email = os.getenv("CONTACT_EMAIL", "anonymous@example.com")

notebook_dir = os.path.dirname(os.path.abspath("__file__"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))

sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py in the config directory.")

# --- 2. Registry & File Setup ---
SOURCE_PREFIX = "LCSH"
registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, 
    config_dir=config_dir,
    fallback_name="Library of Congress Subject Headings",
    fallback_uri="http://id.loc.gov/authorities/subjects/"
)
SOURCE_NAME = registry_data['Source_Name']
BASE_URI = registry_data['Base_URI']
raw_ingest_file = os.path.join(raw_data_dir, f"raw_lcsh_groups.csv") 

HEADERS = {'User-Agent': f'ReligiousMappingProject/1.0 (mailto:{contact_email})', 'Accept': 'application/json'}

# --- 3. Targeted Seeds ---
SEED_URIS = [
    "http://id.loc.gov/authorities/subjects/sh85119451", # Sects
    "http://id.loc.gov/authorities/subjects/sh85034719"  # Cults
]

# --- 4. Constraints ---
MAX_DEPTH = 7

# --- 5. Persistent Progress Tracking & Schema Drift ---
if os.path.exists(raw_ingest_file):
    existing_df = pd.read_csv(raw_ingest_file, dtype={'Concept_ID': str})
    
    if list(existing_df.columns) != COLUMN_ORDER:
        print(f"[!] Schema mismatch detected. Deleting outdated {os.path.basename(raw_ingest_file)}...")
        os.remove(raw_ingest_file)
        visited_uris = set()
    else:
        visited_uris = set(existing_df['URI'].unique().tolist())
        print(f"[*] Resuming: {len(visited_uris)} concepts already in {raw_ingest_file}")
else:
    visited_uris = set()

# --- 6. Extraction Logic ---
def process_lc_node(uri, parent_path="", p_id="", current_depth=0):
    """Recursively crawls with a hard depth limit, grabbing metadata and crosswalks."""
    clean_uri = uri.rstrip('/').replace('.json', '').replace('.rdf', '').replace('.html', '')
    
    if clean_uri in visited_uris:
        return
    visited_uris.add(clean_uri)
    
    time.sleep(0.8) # Politeness
    
    node_data = None
    try:
        res = requests.get(f"{clean_uri}.json", headers=HEADERS, timeout=15)
        if res.status_code == 200:
            data = res.json()
            if isinstance(data, dict): data = [data]
            node_data = next((item for item in data if isinstance(item, dict) and item.get('@id') == clean_uri), {})
    except: pass
            
    if not node_data:
        return

    SKOS_PREF = 'http://www.w3.org/2004/02/skos/core#prefLabel'
    SKOS_ALT  = 'http://www.w3.org/2004/02/skos/core#altLabel'
    SKOS_NARROWER = 'http://www.w3.org/2004/02/skos/core#narrower'
    MADS_CITATION = 'http://www.loc.gov/mads/rdf/v1#citationNote'

    # Label extraction & Translation check
    label = "No Label"
    has_translation = ""
    for pref in node_data.get(SKOS_PREF, []):
        lang = pref.get('@language', '').lower()
        if not lang or lang.startswith('en'):
            label = pref.get('@value', label)
        else:
            has_translation = "yes"
    
    current_path = f"{parent_path} > {label}" if parent_path else label
    
    depth_flag = f"[DEPTH {current_depth}]"
    print(f"\r{depth_flag} Ingested: {label[:60]:<60}", end="", flush=True)

    # Descriptions & Synonyms
    synonyms = []
    for alt in node_data.get(SKOS_ALT, []):
        lang = alt.get('@language', '').lower()
        if not lang or lang.startswith('en'):
            if alt.get('@value'): synonyms.append(alt.get('@value'))
        else:
            has_translation = "yes"

    descriptions = []
    for item in data:
        if isinstance(item, dict) and MADS_CITATION in item:
            for n in item[MADS_CITATION]:
                if n.get('@value'): descriptions.append(n.get('@value'))
                
    # --- UPGRADE: Crosswalk Extraction ---
    crosswalks = []
    for match_prop in [
        'http://www.w3.org/2004/02/skos/core#exactMatch',
        'http://www.w3.org/2004/02/skos/core#closeMatch',
        'http://www.loc.gov/mads/rdf/v1#hasCloseExternalAuthority'
    ]:
        for cw in node_data.get(match_prop, []):
            cw_uri = cw.get('@id')
            if cw_uri: crosswalks.append(cw_uri)
                
    cid = clean_uri.split('/')[-1]

    # Map and Save
    extracted_data = {
        'Source_System': SOURCE_NAME, 
        'Primary_Label': label, 
        'CURIE': f"{SOURCE_PREFIX}:{cid}",
        'Formal_Label': label, 
        'Concept_Type': 'skos:Concept', 
        'Hierarchy_Path': current_path, 
        'Synonyms': " | ".join(list(set(synonyms))), 
        'Description': " | ".join(descriptions), 
        'Parent_IDs': p_id, 
        'Concept_ID': cid, 
        'URI': clean_uri, 
        'Has_Translation': has_translation,
        'Status': 'active',
        'Crosswalks': " | ".join(list(set(crosswalks))) # Add mappings!
    }
    
    clean_row = finalize_row(extracted_data)
    pd.DataFrame([clean_row])[COLUMN_ORDER].to_csv(
        raw_ingest_file, mode='a', index=False, 
        header=not os.path.isfile(raw_ingest_file), encoding='utf-8-sig'
    )

    # RECURSION CHECK
    if current_depth < MAX_DEPTH:
        for child in node_data.get(SKOS_NARROWER, []):
            child_uri = child.get('@id')
            if child_uri and child_uri.startswith(BASE_URI):
                process_lc_node(child_uri, parent_path=current_path, p_id=cid, current_depth=current_depth + 1)
    else:
        print(f"\n[!] LIMIT REACHED: Level {MAX_DEPTH} reached at '{label}'. Pruning children.")

# --- 7. Execution ---
print(f"Starting Depth-Limited Crawl (Limit: {MAX_DEPTH})...\n")
for seed in SEED_URIS:
    process_lc_node(seed, current_depth=0)
print(f"\n\nDone. Rows saved to {raw_ingest_file}")